In [2]:
# import numpy and pandas
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import math

%config Completer.use_jedi=False
import os


In [2]:
dose_response = pd.read_csv(
    "../../output/regression/dose_response_matrix.csv",
    index_col=0
)

/tmp/ipykernel_115708/3951541710.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  dose_response = pd.read_csv(


In [6]:
dose_response.head()

,CELL_LINE_NAME,TCGA_DESC,DRUG_NAME,PATHWAY_NAME,LN_IC50,PUTATIVE_TARGET
750,ES5,UNCLASSIFIED,Sunitinib,RTK signaling,3.773630,NaN
751,ES7,UNCLASSIFIED,Sunitinib,RTK signaling,2.990953,NaN
752,EW-11,UNCLASSIFIED,Sunitinib,RTK signaling,3.600300,NaN
753,SK-ES-1,UNCLASSIFIED,Sunitinib,RTK signaling,0.830185,NaN
754,COLO-829,SKCM,Sunitinib,RTK signaling,5.417977,NaN


In [3]:
expression_data = pd.read_csv(
    "../../output/regression/screendl_GEX_data_filtered_logCPM.csv",
    index_col=0
)

In [8]:
expression_data.head()

,A2M,AAAS,AADAT,AARS1,ABAT,ABCA1,ABCA2,ABCA3,ABCA4,ABCA5,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
model_name,,,,,,,,,,,,,,,,,,,,,
22RV1,4.765043,7.965285,4.980764,10.095432,4.304502,1.847504,9.427391,8.925643,0.165838,6.064003,...,7.342397,2.432863,6.725244,4.657735,6.548773,0.000000,0.0,6.734076,9.082990,6.243274
23132-87,0.064925,6.806492,0.186616,10.601444,2.832288,1.820349,8.509051,4.961153,0.186616,5.428668,...,7.011857,0.064925,6.411042,5.662966,8.226443,0.000000,0.0,6.738466,7.903225,7.250787
42-MG-BA,0.400640,6.799609,3.829836,9.116141,5.224759,3.344072,5.260135,0.596601,1.431086,2.708765,...,6.681754,3.254972,5.330683,4.906913,8.154266,0.045456,0.0,6.610763,7.546943,8.494356
451Lu,10.520395,7.245405,4.697826,9.433920,0.675538,6.452475,7.589598,1.879429,0.215869,5.735919,...,5.959312,4.458321,6.669770,5.332768,7.678688,0.000000,0.0,5.896983,7.514413,8.815926
5637,0.403375,6.791443,5.825079,7.556310,1.249976,6.667477,5.068375,0.371025,0.337933,4.760282,...,6.101350,2.643754,5.801756,4.954274,7.388943,0.000000,0.0,6.634640,7.588321,8.926763


In [3]:
smiles = pd.read_csv(
    "../../output/regression/vector_smiles_512.csv",
    index_col=0
)

In [12]:
smiles.head()

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
Sunitinib,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
PHA-665752,0,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
Cyclopamine,0,0,0,0,0,0,0,1,0,0,...,1,0,1,0,0,1,1,0,0,0
AZ628,0,0,0,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
Imatinib,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1


In [4]:
from pathlib import Path

In [5]:
type = "pancancer"
experiment = "NBS_cells"

In [6]:
train_idx = pd.read_csv(
    f"../../output/regression/cross_validation/{type}/{experiment}/fold_0/train_set.csv",
    index_col=0
)

In [8]:
train_idx.head()

,CELL_LINE_NAME,TCGA_DESC,DRUG_NAME,PATHWAY_NAME,LN_IC50,PUTATIVE_TARGET
108473,HDQ-P1,BRCA,VE-822,Genome integrity,5.070381,ATR
297376,SF268,GBM,AZD8931,RTK signaling,1.226970,NaN
85765,MONO-MAC-6,LAML,FR-180204,ERK MAPK signaling,3.663585,NaN
122001,NCI-H1341,SCLC,Ribociclib,Cell cycle,3.746730,"CDK4, CDK6"
191388,TOV-21G,OV,GNE-317,PI3K/MTOR signaling,0.438354,PI3Kalpha


In [9]:
train_pairs = train_idx[["DRUG_NAME","CELL_LINE_NAME", "LN_IC50"]]

In [10]:
train_pairs.head()

,DRUG_NAME,CELL_LINE_NAME,LN_IC50
108473,VE-822,HDQ-P1,5.070381
297376,AZD8931,SF268,1.226970
85765,FR-180204,MONO-MAC-6,3.663585
122001,Ribociclib,NCI-H1341,3.746730
191388,GNE-317,TOV-21G,0.438354


In [11]:
train_pairs = train_pairs.rename(columns={"LN_IC50":"IC50"})

In [12]:
train_pairs.head()

,DRUG_NAME,CELL_LINE_NAME,IC50
108473,VE-822,HDQ-P1,5.070381
297376,AZD8931,SF268,1.226970
85765,FR-180204,MONO-MAC-6,3.663585
122001,Ribociclib,NCI-H1341,3.746730
191388,GNE-317,TOV-21G,0.438354


In [ ]:
#Make gene expressions grouped into gene sets

In [7]:
#Loading Gene Set
#Gene Set File (gmt) is 'KEGG subset of CP' from MSigDB (http://www.gsea-msigdb.org/gsea/msigdb/collections.jsp)
GeneSet_List=[]
GeneSetFile='RawFile/c2.cp.kegg_legacy.v2024.1.Hs.symbols.gmt'
with open(GeneSetFile) as f:
    reader = csv.reader(f)
    data = list(list(rec) for rec in csv.reader(f, delimiter='\t')) #reads csv into a list of lists
    for row in data:
        GeneSet_List.append(row)

GeneSet_Dic={}
for GeneSet in GeneSet_List:
    GeneSet_Dic[GeneSet[0]]=GeneSet[2:]

#Delete genes that are not valid
#In here, E3 is just a name of one of cell line that is valid
#GeneSet_Dic_withoutNA={}
#for GeneSet in GeneSet_Dic:
#    GeneSet_Dic_withoutNA[GeneSet]=expression_df[GeneSet_Dic[GeneSet]].dropna().index.values


In [10]:
GeneSet_Dic.keys()

dict_keys(['KEGG_ABC_TRANSPORTERS', 'KEGG_ACUTE_MYELOID_LEUKEMIA', 'KEGG_ADHERENS_JUNCTION', 'KEGG_ADIPOCYTOKINE_SIGNALING_PATHWAY', 'KEGG_ALANINE_ASPARTATE_AND_GLUTAMATE_METABOLISM', 'KEGG_ALDOSTERONE_REGULATED_SODIUM_REABSORPTION', 'KEGG_ALLOGRAFT_REJECTION', 'KEGG_ALPHA_LINOLENIC_ACID_METABOLISM', 'KEGG_ALZHEIMERS_DISEASE', 'KEGG_AMINOACYL_TRNA_BIOSYNTHESIS', 'KEGG_AMINO_SUGAR_AND_NUCLEOTIDE_SUGAR_METABOLISM', 'KEGG_AMYOTROPHIC_LATERAL_SCLEROSIS_ALS', 'KEGG_ANTIGEN_PROCESSING_AND_PRESENTATION', 'KEGG_APOPTOSIS', 'KEGG_ARACHIDONIC_ACID_METABOLISM', 'KEGG_ARGININE_AND_PROLINE_METABOLISM', 'KEGG_ARRHYTHMOGENIC_RIGHT_VENTRICULAR_CARDIOMYOPATHY_ARVC', 'KEGG_ASCORBATE_AND_ALDARATE_METABOLISM', 'KEGG_ASTHMA', 'KEGG_AUTOIMMUNE_THYROID_DISEASE', 'KEGG_AXON_GUIDANCE', 'KEGG_BASAL_CELL_CARCINOMA', 'KEGG_BASAL_TRANSCRIPTION_FACTORS', 'KEGG_BASE_EXCISION_REPAIR', 'KEGG_BETA_ALANINE_METABOLISM', 'KEGG_BIOSYNTHESIS_OF_UNSATURATED_FATTY_ACIDS', 'KEGG_BLADDER_CANCER', 'KEGG_BUTANOATE_METABOLISM', 'K

In [11]:
gn = expression_data.columns
gn

Index(['A2M', 'AAAS', 'AADAT', 'AARS1', 'ABAT', 'ABCA1', 'ABCA2', 'ABCA3',
       'ABCA4', 'ABCA5',
       ...
       'ZNF292', 'ZNF365', 'ZNF639', 'ZNF707', 'ZNFX1', 'ZNRF4', 'ZPBP',
       'ZW10', 'ZWINT', 'ZYX'],
      dtype='object', length=4377)

In [12]:
GeneSet_Dic_withoutNA={}
for GeneSet in GeneSet_Dic:
    GeneSet_Dic_withoutNA[GeneSet] = gn.intersection(GeneSet_Dic[GeneSet]).to_list()

# Extract cell line features

Produces a GEX matrix of [genes x cell lines] that will be the input for the model

In [19]:
def CelllineFeatureExtract(ExpressionMatrix, GeneSetDic, CellLine):
    X_Feature=[]
    for GeneSet in GeneSetDic.keys():
        Gene_in_GeneSet=[]
        for Gene in GeneSetDic[GeneSet]:
            Gene_in_GeneSet.append(Gene)
        X_Feature.append(ExpressionMatrix[Gene_in_GeneSet].loc[[CellLine]])
    
    return X_Feature

In [ ]:
### original
cellline_input=[]

for i in range(len(GeneSet_Dic_withoutNA)):
    cellline_input.append([])
    
for cellline in expression_df.index.to_list():
    x=CelllineFeatureExtract(expression_df, GeneSet_Dic_withoutNA, cellline)
    for j in range(len(GeneSet_Dic_withoutNA)):
        cellline_input[j].append(x[j])

In [ ]:
cellline_input = [pd.concat(g, axis=0) for g in cellline_input]

In [ ]:
for idx,df in enumerate(cellline_input):
    df.to_csv('ProcessedFile/CellLine/'+str(idx)+'.csv')

In [19]:
expression_data.head()

,A2M,AAAS,AADAT,AARS1,ABAT,ABCA1,ABCA2,ABCA3,ABCA4,ABCA5,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
model_name,,,,,,,,,,,,,,,,,,,,,
22RV1,4.765043,7.965285,4.980764,10.095432,4.304502,1.847504,9.427391,8.925643,0.165838,6.064003,...,7.342397,2.432863,6.725244,4.657735,6.548773,0.000000,0.0,6.734076,9.082990,6.243274
23132-87,0.064925,6.806492,0.186616,10.601444,2.832288,1.820349,8.509051,4.961153,0.186616,5.428668,...,7.011857,0.064925,6.411042,5.662966,8.226443,0.000000,0.0,6.738466,7.903225,7.250787
42-MG-BA,0.400640,6.799609,3.829836,9.116141,5.224759,3.344072,5.260135,0.596601,1.431086,2.708765,...,6.681754,3.254972,5.330683,4.906913,8.154266,0.045456,0.0,6.610763,7.546943,8.494356
451Lu,10.520395,7.245405,4.697826,9.433920,0.675538,6.452475,7.589598,1.879429,0.215869,5.735919,...,5.959312,4.458321,6.669770,5.332768,7.678688,0.000000,0.0,5.896983,7.514413,8.815926
5637,0.403375,6.791443,5.825079,7.556310,1.249976,6.667477,5.068375,0.371025,0.337933,4.760282,...,6.101350,2.643754,5.801756,4.954274,7.388943,0.000000,0.0,6.634640,7.588321,8.926763


In [22]:
pathway_names = list(GeneSet_Dic_withoutNA.keys())

In [26]:
train_gex = expression_data.loc[train_idx.CELL_LINE_NAME]

In [45]:
#################
# CUSTOM
#################
cellline_input = [
        expression_data[GeneSet_Dic_withoutNA[path]]  # Subset all cell lines at once for this gene set
        for path in pathway_names
    ]
# for idx, df in enumerate(cellline_input):
#     df.to_csv(f'input/{idx}.csv')

In [49]:
cellline_input[0].loc[train_idx.iloc[0:10].CELL_LINE_NAME]

,ABCA1,ABCA2,ABCA3,ABCA4,ABCA5,ABCA6,ABCA8,ABCA9,ABCB1,ABCB11,...,ABCC8,ABCD1,ABCD2,ABCD3,ABCG2,ABCG4,ABCG8,CFTR,TAP1,TAP2
model_name,,,,,,,,,,,,,,,,,,,,,
HDQ-P1,4.386648,6.279881,6.485041,7.738326,5.813840,0.252798,0.131929,0.067472,0.067472,0.000000,...,0.516933,5.158346,0.000000,7.574046,5.708141,0.610387,0.364319,0.131929,9.068791,9.285625
SF268,3.927118,6.003559,7.351364,1.540003,4.322370,0.034865,1.426427,0.034865,0.315693,0.710158,...,4.782179,5.466706,0.166464,6.738679,0.710158,3.687957,0.000000,0.287056,7.539901,7.254063
MONO-MAC-6,3.387834,8.682171,4.935469,0.000000,5.394961,0.345394,0.300833,0.430588,0.207356,0.054699,...,0.000000,6.791599,1.859052,7.523346,0.471370,0.300833,0.054699,0.254851,8.089266,8.861747
NCI-H1341,1.420788,8.025102,8.821334,3.273957,6.060537,0.000000,0.000000,0.057846,0.000000,2.668853,...,1.045208,5.346349,0.167012,8.706926,0.408474,1.257581,0.316680,0.452270,5.921392,7.166411
TOV-21G,2.320911,7.979162,9.018010,0.621853,4.162226,0.000000,0.063371,0.000000,2.433124,0.063371,...,2.359293,5.029342,0.063371,7.470731,0.292218,1.285068,0.182327,0.344176,7.578437,6.548529
H3255,3.219019,7.033327,6.104374,6.163124,4.018512,0.000000,6.703757,0.038050,0.038050,1.721751,...,0.075122,3.271868,0.592424,7.640351,4.363841,0.180945,0.111266,0.827844,7.075130,7.555400
Raji,4.284780,3.675978,1.014669,0.086573,3.853122,4.057167,0.086573,0.549754,2.741946,0.000000,...,0.168244,1.977327,0.969819,7.110965,0.086573,0.000000,0.086573,0.127986,7.750795,9.580772
G-401,6.480907,6.244318,5.903672,0.165698,6.561196,0.824417,0.000000,1.010233,0.405508,0.000000,...,2.314930,3.918212,0.000000,7.212450,9.156324,2.936099,0.405508,0.216955,6.216882,6.524982
D-502MG,9.824248,8.712445,7.857826,0.235438,2.318589,0.000000,0.000000,0.000000,5.198380,0.141961,...,0.062558,7.212150,0.822236,7.449013,0.722438,3.592664,0.306090,0.000000,8.549344,6.467166


In [ ]:
type = "pancancer"
experiment = "NBS_cells"

for n in range(1,10):
    d = f"CV/{type}/{experiment}/fold_{n}"
    train_dir = f"{d}/training/"
    test_dir = f"{d}/testing/"
    

    Path(f"{train_dir}/input/").mkdir(parents=True, exist_ok=True)
    Path(f"{test_dir}/input/").mkdir(parents=True, exist_ok=True)
    
    train_idx = pd.read_csv(
        f"../../output/regression/cross_validation/{type}/{experiment}/fold_{n}/train_set.csv",
        index_col=0
    )
    
    train_pairs = train_idx[["DRUG_NAME", "CELL_LINE_NAME", "LN_IC50"]]
    train_pairs = train_pairs.rename(columns={"LN_IC50":"IC50"})

    train_gex = expression_data.loc[train_idx.CELL_LINE_NAME]
    gex_mean = train_gex.mean(0)
    gex_std = train_gex.std(0)
    train_gex = (train_gex-gex_mean)/gex_std

    train_drug = smiles.loc[train_idx.DRUG_NAME]

    assert(len(train_pairs)==len(train_gex))

    train_pairs.to_csv(f"{train_dir}/Training.csv")
    train_drug.to_csv(f"{train_dir}/input/drug.csv", index=True)

    
    print("Entering cell feature extraction for Training")

    ### custom
    cellline_input = [
        train_gex[genes]  # Subset all cell lines at once for this gene set
        for genes in GeneSet_Dic_withoutNA.values()
    ]
    for idx, df in enumerate(cellline_input):
        df.to_csv(f"{train_dir}/input/{idx}.csv")
    
    #index.rename('Gene_Symbol')
    train_gex = train_gex.T
    train_gex.index = train_gex.index.rename("Gene_Symbol")
    train_gex.to_csv(f"{train_dir}/expression.csv")
    
    ## test
    test_idx = pd.read_csv(
        f"../../output/regression/cross_validation/{type}/{experiment}/fold_{n}/test_set.csv",
        index_col=0
    )
    test_pairs = test_idx[["DRUG_NAME", "CELL_LINE_NAME", "LN_IC50"]]
    test_pirs = test_pairs.rename(columns={"LN_IC50": "IC50"})
    test_pairs.to_csv(f"{test_dir}/Testing.csv")
    
    test_gex = expression_data.loc[test_idx.CELL_LINE_NAME]
    test_gex = (test_gex - gex_mean) / gex_std

    
    test_drug = smiles.loc[test_idx.DRUG_NAME]
    test_drug.to_csv(f"{test_dir}/input/drug.csv", index=False)

    print("Entering cell feature extraction for Testing")
    ### custom
    cellline_input = [
        test_gex[genes]  # Subset all cell lines at once for this gene set
        for genes in GeneSet_Dic_withoutNA.values()
    ]
    
    
    for idx, df in enumerate(cellline_input):
        df.to_csv(f"{test_dir}/input/{idx}.csv")

    test_gex = test_gex.T
    test_gex.index = test_gex.index.rename("Gene_Symbol")
    test_gex.to_csv(f"{test_dir}/expression.csv")

    print(f"✓ Finished fold {n}: {train_pairs.shape[0]} train pairs, {test_pairs.shape[0]} test pairs")

    

In [ ]:
#Loading drug information
#It is attached to the source code and SMILESs of each drug were manually collected from DrugBank and PubChem
#Morgan fingerprint was calculated by using RDKit
#The drug information with Morgan fingerprint can be used as the input feature directly
drug_df=pd.read_csv('ProcessedFile/Drug.csv',index_col=0)

In [ ]:
drug_list=drug_df.index

In [ ]:
#GDSC response data preprocessing

In [ ]:
#Loading GDSC response data
#GDSC response data is 'GDSC1-dataset'
#from GDSC Downloads pages (https://www.cancerrxgene.org/downloads/bulk_download)
GDSC_response=pd.read_excel('RawFile/v17.3_fitted_dose_response.xlsx')
GDSC_response=GDSC_response[['DRUG_NAME','CELL_LINE_NAME','LN_IC50']]
GDSC_response=GDSC_response.reset_index()
GDSC_response.columns=['Origin_idx','Drug name','Cell line name','IC50']

In [ ]:
#Excluding cell line-drug pair whose cell line information or drug information is not valid
cellline_in_GDSC=GDSC_response['Cell line name']
drug_in_GDSC=GDSC_response['Drug name']
is_valid_cellline=[(cellline in expression_df.columns) for cellline in cellline_in_GDSC]
is_valid_drug=[(drug in drug_list) for drug in drug_in_GDSC]
is_valid_all=[(cellline_validity&drug_validaity) for cellline_validity,drug_validaity in zip(is_valid_cellline,is_valid_drug)]
GDSC_response=GDSC_response.loc[is_valid_all]


In [ ]:
GDSC_response.to_csv('GDSC_response.csv',index=False)